# 08a - Advanced Feature Enhancement for Packet Images

This notebook extends the feature engineering pipeline (notebook 08) by exploring advanced feature enhancement techniques including wavelet transforms, frequency domain analysis, and statistical features to enrich packet image representations before feeding them to Vision Transformers.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Signal processing
import pywt  # PyWavelets for wavelet transforms
from scipy import signal, fft, ndimage
from scipy.stats import entropy, kurtosis, skew
from skimage import feature, filters

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F

# Visualization
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.cm as cm

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')

# Configure notebook display
pd.set_option('display.max_columns', None)

In [ ]:
# Setup project paths
notebook_path = Path().resolve()
project_root = notebook_path.parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our modules
from src.data.packet_to_image import PacketImageEncoder

# Define paths
data_dir = project_root / 'data'
raw_data_dir = data_dir / 'raw'
figures_dir = project_root / 'notebooks' / 'figures'

print(f"Project root: {project_root}")

In [ ]:
# Load sample data
cic_data = pd.read_csv(raw_data_dir / 'payload_byte' / 'cic_ids2017_sample.csv')
byte_columns = [f'byte_{i}' for i in range(1500)]

# Get samples from different classes
samples = {}
for label in range(6):
    if label == 0:
        samples['benign'] = cic_data[cic_data['label'] == label][byte_columns].values[:5]
    else:
        samples[f'attack_{label}'] = cic_data[cic_data['label'] == label][byte_columns].values[:5]

# Create packet images
encoder = PacketImageEncoder(image_size=(128, 128))
sample_images = {}
for class_name, packets in samples.items():
    sample_images[class_name] = [encoder.encode_sequential(packet) for packet in packets]

print(f"Loaded {len(samples)} classes with {len(sample_images['benign'])} samples each")

## 1. Wavelet Transform Features

Wavelet transforms can capture both frequency and spatial information, making them ideal for analyzing packet data patterns at multiple scales.

In [ ]:
class WaveletFeatureExtractor:
    """Extract wavelet-based features from packet images."""
    
    def __init__(self, wavelet='db4', levels=4):
        """
        Initialize wavelet feature extractor.
        
        Args:
            wavelet: Wavelet type (e.g., 'db4', 'sym5', 'coif3')
            levels: Number of decomposition levels
        """
        self.wavelet = wavelet
        self.levels = levels
        
    def extract_2d_dwt_features(self, image):
        """
        Extract 2D Discrete Wavelet Transform features.
        
        Returns approximation and detail coefficients.
        """
        # Ensure image is float
        image = image.astype(np.float32)
        
        # Perform 2D DWT
        coeffs = pywt.wavedec2(image, self.wavelet, level=self.levels)
        
        # Extract features from each level
        features = {
            'approximation': coeffs[0],
            'details': []
        }
        
        for level in range(1, len(coeffs)):
            # Each level has (horizontal, vertical, diagonal) details
            cH, cV, cD = coeffs[level]
            features['details'].append({
                'horizontal': cH,
                'vertical': cV,
                'diagonal': cD
            })
            
        return features
    
    def extract_wavelet_energy(self, image):
        """
        Extract energy features from wavelet decomposition.
        
        Energy is computed as the sum of squared coefficients.
        """
        features = self.extract_2d_dwt_features(image)
        
        # Compute energy for approximation
        energy_features = {
            'approx_energy': np.sum(features['approximation'] ** 2)
        }
        
        # Compute energy for each detail level
        for i, detail in enumerate(features['details']):
            energy_features[f'level_{i+1}_horizontal_energy'] = np.sum(detail['horizontal'] ** 2)
            energy_features[f'level_{i+1}_vertical_energy'] = np.sum(detail['vertical'] ** 2)
            energy_features[f'level_{i+1}_diagonal_energy'] = np.sum(detail['diagonal'] ** 2)
            
        # Normalize by total energy
        total_energy = sum(energy_features.values())
        for key in energy_features:
            energy_features[key] /= total_energy
            
        return energy_features
    
    def extract_wavelet_statistics(self, image):
        """
        Extract statistical features from wavelet coefficients.
        """
        features = self.extract_2d_dwt_features(image)
        stats = {}
        
        # Statistics for approximation coefficients
        approx = features['approximation'].flatten()
        stats['approx_mean'] = np.mean(approx)
        stats['approx_std'] = np.std(approx)
        stats['approx_entropy'] = entropy(np.histogram(approx, bins=50)[0] + 1e-10)
        
        # Statistics for detail coefficients
        for i, detail in enumerate(features['details']):
            for direction, coeffs in detail.items():
                coeffs_flat = coeffs.flatten()
                prefix = f'level_{i+1}_{direction}'
                
                stats[f'{prefix}_mean'] = np.mean(coeffs_flat)
                stats[f'{prefix}_std'] = np.std(coeffs_flat)
                stats[f'{prefix}_skew'] = skew(coeffs_flat)
                stats[f'{prefix}_kurtosis'] = kurtosis(coeffs_flat)
                
        return stats

# Test wavelet feature extraction
wavelet_extractor = WaveletFeatureExtractor()

# Extract features from a sample image
sample_img = sample_images['benign'][0]
wavelet_features = wavelet_extractor.extract_wavelet_energy(sample_img)
wavelet_stats = wavelet_extractor.extract_wavelet_statistics(sample_img)

print("Wavelet Energy Features:")
for key, value in wavelet_features.items():
    print(f"  {key}: {value:.4f}")
    
print(f"\nTotal wavelet statistics features: {len(wavelet_stats)}")

In [ ]:
# Visualize wavelet decomposition
def visualize_wavelet_decomposition(image, wavelet='db4', levels=3):
    """Visualize multi-level wavelet decomposition."""
    coeffs = pywt.wavedec2(image, wavelet, level=levels)
    
    # Determine figure layout
    n_levels = len(coeffs) - 1
    fig, axes = plt.subplots(n_levels + 1, 4, figsize=(15, 4 * (n_levels + 1)))
    
    # Plot approximation
    axes[0, 0].imshow(coeffs[0], cmap='gray')
    axes[0, 0].set_title(f'Approximation (Level {levels})')
    axes[0, 0].axis('off')
    
    # Hide unused subplots in first row
    for j in range(1, 4):
        axes[0, j].axis('off')
    
    # Plot details at each level
    for i in range(1, len(coeffs)):
        cH, cV, cD = coeffs[i]
        
        axes[i, 0].imshow(np.abs(cH), cmap='gray')
        axes[i, 0].set_title(f'Horizontal Details (Level {levels - i + 1})')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(np.abs(cV), cmap='gray')
        axes[i, 1].set_title(f'Vertical Details (Level {levels - i + 1})')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(np.abs(cD), cmap='gray')
        axes[i, 2].set_title(f'Diagonal Details (Level {levels - i + 1})')
        axes[i, 2].axis('off')
        
        # Combined visualization
        combined = np.sqrt(cH**2 + cV**2 + cD**2)
        axes[i, 3].imshow(combined, cmap='hot')
        axes[i, 3].set_title(f'Combined Energy (Level {levels - i + 1})')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'Wavelet Decomposition ({wavelet})', fontsize=16)
    plt.tight_layout()
    return fig

# Visualize for benign and attack samples
fig = visualize_wavelet_decomposition(sample_images['benign'][0])
plt.savefig(figures_dir / 'wavelet_decomposition_benign.png', dpi=300, bbox_inches='tight')
plt.show()

fig = visualize_wavelet_decomposition(sample_images['attack_1'][0])
plt.savefig(figures_dir / 'wavelet_decomposition_attack.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Frequency Domain Features

Frequency domain analysis can reveal periodic patterns and regularities in packet data that might indicate specific protocols or attack behaviors.

In [ ]:
class FrequencyDomainExtractor:
    """Extract frequency domain features from packet images."""
    
    def extract_fft_features(self, image):
        """
        Extract 2D FFT features from packet image.
        """
        # Compute 2D FFT
        f_transform = fft.fft2(image)
        f_shift = fft.fftshift(f_transform)
        
        # Magnitude spectrum
        magnitude_spectrum = np.abs(f_shift)
        
        # Power spectrum
        power_spectrum = magnitude_spectrum ** 2
        
        # Phase spectrum
        phase_spectrum = np.angle(f_shift)
        
        return {
            'magnitude': magnitude_spectrum,
            'power': power_spectrum,
            'phase': phase_spectrum,
            'fft_shift': f_shift
        }
    
    def extract_frequency_statistics(self, image):
        """
        Extract statistical features from frequency domain.
        """
        fft_features = self.extract_fft_features(image)
        magnitude = fft_features['magnitude']
        
        # Get image dimensions
        h, w = image.shape
        center_h, center_w = h // 2, w // 2
        
        # Create radial bins for frequency analysis
        y, x = np.ogrid[:h, :w]
        r = np.sqrt((x - center_w)**2 + (y - center_h)**2)
        
        # Compute radial average
        r_max = min(center_h, center_w)
        n_bins = 50
        radial_bins = np.linspace(0, r_max, n_bins)
        radial_profile = []
        
        for i in range(len(radial_bins) - 1):
            mask = (r >= radial_bins[i]) & (r < radial_bins[i + 1])
            radial_profile.append(np.mean(magnitude[mask]))
        
        radial_profile = np.array(radial_profile)
        
        # Extract features
        features = {
            'dc_component': magnitude[center_h, center_w],
            'total_energy': np.sum(fft_features['power']),
            'low_freq_energy': np.sum(fft_features['power'][r < r_max * 0.1]),
            'high_freq_energy': np.sum(fft_features['power'][r > r_max * 0.9]),
            'radial_mean': np.mean(radial_profile),
            'radial_std': np.std(radial_profile),
            'spectral_centroid': np.sum(radial_bins[:-1] * radial_profile) / np.sum(radial_profile)
        }
        
        # Normalize energies
        features['low_freq_ratio'] = features['low_freq_energy'] / features['total_energy']
        features['high_freq_ratio'] = features['high_freq_energy'] / features['total_energy']
        
        return features
    
    def extract_dct_features(self, image):
        """
        Extract Discrete Cosine Transform features.
        DCT is often used in compression and can capture important patterns.
        """
        from scipy.fftpack import dct
        
        # Compute 2D DCT
        dct_coeffs = dct(dct(image.T, norm='ortho').T, norm='ortho')
        
        # Extract features from DCT coefficients
        features = {
            'dct_energy': np.sum(dct_coeffs ** 2),
            'dct_entropy': entropy(np.abs(dct_coeffs).flatten() + 1e-10)
        }
        
        # Zigzag scan of top-left coefficients (like JPEG)
        n = 8  # Size of block to analyze
        zigzag_coeffs = []
        
        for i in range(2 * n - 1):
            if i < n:
                for j in range(i + 1):
                    if i - j < image.shape[0] and j < image.shape[1]:
                        zigzag_coeffs.append(dct_coeffs[i - j, j])
            else:
                for j in range(i - n + 1, n):
                    if i - j < image.shape[0] and j < image.shape[1]:
                        zigzag_coeffs.append(dct_coeffs[i - j, j])
        
        zigzag_coeffs = np.array(zigzag_coeffs[:64])  # Keep first 64 coefficients
        
        features['dct_zigzag_mean'] = np.mean(np.abs(zigzag_coeffs))
        features['dct_zigzag_std'] = np.std(np.abs(zigzag_coeffs))
        
        return features
    
    def create_frequency_enhanced_image(self, image):
        """
        Create enhanced image using frequency domain filtering.
        """
        # Get FFT
        fft_features = self.extract_fft_features(image)
        f_shift = fft_features['fft_shift']
        
        h, w = image.shape
        center_h, center_w = h // 2, w // 2
        
        # Create multiple frequency bands
        y, x = np.ogrid[:h, :w]
        r = np.sqrt((x - center_w)**2 + (y - center_h)**2)
        
        # Low-pass filter
        low_pass_mask = r <= min(center_h, center_w) * 0.3
        low_freq = f_shift * low_pass_mask
        
        # Band-pass filter
        band_pass_mask = (r > min(center_h, center_w) * 0.3) & (r <= min(center_h, center_w) * 0.7)
        mid_freq = f_shift * band_pass_mask
        
        # High-pass filter
        high_pass_mask = r > min(center_h, center_w) * 0.7
        high_freq = f_shift * high_pass_mask
        
        # Inverse transform for each band
        low_img = np.real(fft.ifft2(fft.ifftshift(low_freq)))
        mid_img = np.real(fft.ifft2(fft.ifftshift(mid_freq)))
        high_img = np.real(fft.ifft2(fft.ifftshift(high_freq)))
        
        # Stack as multi-channel image
        enhanced = np.stack([low_img, mid_img, high_img], axis=-1)
        
        return enhanced

# Test frequency domain extraction
freq_extractor = FrequencyDomainExtractor()

# Extract features
sample_img = sample_images['benign'][0]
freq_stats = freq_extractor.extract_frequency_statistics(sample_img)
dct_features = freq_extractor.extract_dct_features(sample_img)

print("Frequency Domain Features:")
for key, value in freq_stats.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:.4f}")
        
print("\nDCT Features:")
for key, value in dct_features.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Visualize frequency domain analysis
def visualize_frequency_analysis(image):
    """Visualize frequency domain components."""
    freq_extractor = FrequencyDomainExtractor()
    fft_features = freq_extractor.extract_fft_features(image)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    axes[0, 0].imshow(image, cmap='gray')
    axes[0, 0].set_title('Original Packet Image')
    axes[0, 0].axis('off')
    
    # Magnitude spectrum (log scale)
    magnitude_log = np.log(fft_features['magnitude'] + 1)
    im = axes[0, 1].imshow(magnitude_log, cmap='hot')
    axes[0, 1].set_title('Magnitude Spectrum (log scale)')
    axes[0, 1].axis('off')
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046)
    
    # Phase spectrum
    im = axes[0, 2].imshow(fft_features['phase'], cmap='hsv')
    axes[0, 2].set_title('Phase Spectrum')
    axes[0, 2].axis('off')
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046)
    
    # Frequency bands
    enhanced = freq_extractor.create_frequency_enhanced_image(image)
    
    axes[1, 0].imshow(enhanced[:, :, 0], cmap='gray')
    axes[1, 0].set_title('Low Frequency Components')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(enhanced[:, :, 1], cmap='gray')
    axes[1, 1].set_title('Mid Frequency Components')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(enhanced[:, :, 2], cmap='gray')
    axes[1, 2].set_title('High Frequency Components')
    axes[1, 2].axis('off')
    
    plt.suptitle('Frequency Domain Analysis', fontsize=16)
    plt.tight_layout()
    return fig

# Visualize for different traffic types
fig = visualize_frequency_analysis(sample_images['benign'][0])
plt.savefig(figures_dir / 'frequency_analysis_benign.png', dpi=300, bbox_inches='tight')
plt.show()

fig = visualize_frequency_analysis(sample_images['attack_1'][0])
plt.savefig(figures_dir / 'frequency_analysis_attack.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Statistical and Texture Features

Extract statistical and texture-based features that can capture local patterns and distributions in packet data.

In [ ]:
class StatisticalFeatureExtractor:
    """Extract statistical and texture features from packet images."""
    
    def extract_histogram_features(self, image, n_bins=50):
        """
        Extract features from intensity histogram.
        """
        # Compute histogram
        hist, bin_edges = np.histogram(image.flatten(), bins=n_bins, range=(0, 255))
        hist = hist / hist.sum()  # Normalize
        
        # Find peaks
        from scipy.signal import find_peaks
        peaks, properties = find_peaks(hist, height=0.01)
        
        features = {
            'hist_entropy': entropy(hist + 1e-10),
            'hist_energy': np.sum(hist ** 2),
            'hist_uniformity': np.sum(hist ** 2),
            'hist_max': np.max(hist),
            'hist_mean': np.sum(bin_edges[:-1] * hist),
            'hist_std': np.sqrt(np.sum((bin_edges[:-1] - np.sum(bin_edges[:-1] * hist))**2 * hist)),
            'hist_skewness': skew(image.flatten()),
            'hist_kurtosis': kurtosis(image.flatten()),
            'num_peaks': len(peaks),
            'dominant_intensity': bin_edges[np.argmax(hist)]
        }
        
        return features
    
    def extract_glcm_features(self, image, distances=[1, 3, 5], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
        """
        Extract Gray Level Co-occurrence Matrix (GLCM) features.
        GLCM captures spatial relationships between pixel intensities.
        """
        from skimage.feature import graycomatrix, graycoprops
        
        # Quantize image to fewer gray levels for GLCM
        image_quantized = (image / 4).astype(np.uint8)  # 64 gray levels
        
        # Compute GLCM
        glcm = graycomatrix(image_quantized, distances=distances, angles=angles,
                           levels=64, symmetric=True, normed=True)
        
        # Extract properties
        features = {}
        properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
        
        for prop in properties:
            values = graycoprops(glcm, prop)
            features[f'glcm_{prop}_mean'] = np.mean(values)
            features[f'glcm_{prop}_std'] = np.std(values)
            features[f'glcm_{prop}_range'] = np.ptp(values)
        
        return features
    
    def extract_lbp_features(self, image, radius=3, n_points=24):
        """
        Extract Local Binary Pattern features.
        LBP captures local texture patterns.
        """
        from skimage.feature import local_binary_pattern
        
        # Compute LBP
        lbp = local_binary_pattern(image, n_points, radius, method='uniform')
        
        # Compute histogram of LBP
        n_bins = n_points + 2  # Number of uniform patterns + 2
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
        hist = hist.astype(float) / hist.sum()
        
        features = {
            'lbp_entropy': entropy(hist + 1e-10),
            'lbp_energy': np.sum(hist ** 2),
            'lbp_uniformity': np.sum(hist ** 2)
        }
        
        # Add individual bin values as features
        for i in range(min(10, len(hist))):  # First 10 bins
            features[f'lbp_bin_{i}'] = hist[i]
        
        return features

# Test statistical feature extraction
stat_extractor = StatisticalFeatureExtractor()

# Extract all statistical features
sample_img = sample_images['benign'][0]
hist_features = stat_extractor.extract_histogram_features(sample_img)
glcm_features = stat_extractor.extract_glcm_features(sample_img)
lbp_features = stat_extractor.extract_lbp_features(sample_img)

print(f"Histogram features: {len(hist_features)}")
print(f"GLCM features: {len(glcm_features)}")
print(f"LBP features: {len(lbp_features)}")

# Sample some features
print("\nSample features:")
for key in list(hist_features.keys())[:5]:
    print(f"  {key}: {hist_features[key]:.4f}")

## 4. PyTorch Feature Enhancement Module

Create a PyTorch module that combines all enhancement techniques for integration with Vision Transformers.

In [ ]:
class FeatureEnhancementModule(nn.Module):
    """
    PyTorch module for feature enhancement of packet images.
    Combines wavelet, frequency domain, and statistical features.
    """
    
    def __init__(self, 
                 image_size: int = 128,
                 enhancement_types: list = ['wavelet', 'frequency', 'statistical']):
        super().__init__()
        
        self.image_size = image_size
        self.enhancement_types = enhancement_types
        
        # Initialize extractors
        self.wavelet_extractor = WaveletFeatureExtractor()
        self.freq_extractor = FrequencyDomainExtractor()
        self.stat_extractor = StatisticalFeatureExtractor()
        
        # Feature processing layers
        if 'wavelet' in enhancement_types:
            self.wavelet_proj = nn.Sequential(
                nn.Linear(13, 64),  # 13 wavelet energy features
                nn.ReLU(),
                nn.Dropout(0.1)
            )
            
        if 'frequency' in enhancement_types:
            self.freq_proj = nn.Sequential(
                nn.Linear(9, 64),  # 9 frequency features
                nn.ReLU(), 
                nn.Dropout(0.1)
            )
            
        if 'statistical' in enhancement_types:
            self.stat_proj = nn.Sequential(
                nn.Linear(10, 64),  # Sample of statistical features
                nn.ReLU(),
                nn.Dropout(0.1)
            )
        
        # Fusion layer
        total_features = len(enhancement_types) * 64
        self.fusion = nn.Sequential(
            nn.Linear(total_features, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )
        
    def extract_features(self, image_np):
        """Extract all features from numpy image."""
        features = []
        
        if 'wavelet' in self.enhancement_types:
            wavelet_features = self.wavelet_extractor.extract_wavelet_energy(image_np)
            features.extend(list(wavelet_features.values()))
            
        if 'frequency' in self.enhancement_types:
            freq_features = self.freq_extractor.extract_frequency_statistics(image_np)
            features.extend(list(freq_features.values()))
            
        if 'statistical' in self.enhancement_types:
            hist_features = self.stat_extractor.extract_histogram_features(image_np)
            # Use subset of histogram features
            selected_features = ['hist_entropy', 'hist_energy', 'hist_mean', 'hist_std',
                               'hist_skewness', 'hist_kurtosis', 'hist_max', 'hist_uniformity',
                               'num_peaks', 'dominant_intensity']
            features.extend([hist_features[key] for key in selected_features])
            
        return np.array(features, dtype=np.float32)
    
    def forward(self, images):
        """
        Forward pass for batch of images.
        
        Args:
            images: Tensor of shape (B, C, H, W)
            
        Returns:
            Enhanced features of shape (B, 128)
        """
        batch_size = images.shape[0]
        device = images.device
        
        # Extract features for each image in batch
        batch_features = []
        
        for i in range(batch_size):
            # Convert to numpy for feature extraction
            img_np = images[i, 0].cpu().numpy()  # Remove channel dim and move to CPU
            features = self.extract_features(img_np)
            batch_features.append(features)
            
        # Convert to tensor
        batch_features = torch.tensor(batch_features, device=device)
        
        # Process different feature types
        processed_features = []
        feature_idx = 0
        
        if 'wavelet' in self.enhancement_types:
            wavelet_feat = batch_features[:, feature_idx:feature_idx+13]
            processed_features.append(self.wavelet_proj(wavelet_feat))
            feature_idx += 13
            
        if 'frequency' in self.enhancement_types:
            freq_feat = batch_features[:, feature_idx:feature_idx+9]
            processed_features.append(self.freq_proj(freq_feat))
            feature_idx += 9
            
        if 'statistical' in self.enhancement_types:
            stat_feat = batch_features[:, feature_idx:feature_idx+10]
            processed_features.append(self.stat_proj(stat_feat))
            feature_idx += 10
        
        # Concatenate and fuse features
        combined_features = torch.cat(processed_features, dim=1)
        enhanced_features = self.fusion(combined_features)
        
        return enhanced_features

# Test the enhancement module
enhancement_module = FeatureEnhancementModule()

# Create dummy batch
dummy_images = torch.randn(4, 1, 128, 128)  # Batch of 4 grayscale 128x128 images
enhanced_features = enhancement_module(dummy_images)

print(f"Input shape: {dummy_images.shape}")
print(f"Enhanced features shape: {enhanced_features.shape}")
print(f"Feature enhancement module parameters: {sum(p.numel() for p in enhancement_module.parameters())}")

## 5. Summary and Integration Guidelines

This notebook demonstrated advanced feature enhancement techniques for packet images:

1. **Wavelet Features**: Capture multi-scale spatial-frequency information
2. **Frequency Domain Features**: Reveal periodic patterns and spectral characteristics  
3. **Statistical Features**: Extract texture and distribution properties
4. **PyTorch Integration**: Modular design for easy ViT integration

These enhancement techniques can be integrated with Vision Transformers in several ways:
- As additional input channels to patch embeddings
- As auxiliary features fused with ViT embeddings
- As preprocessing steps to create enhanced packet images

The modular design allows flexible experimentation with different combinations of enhancement techniques.